# Trabalho 1 - Classificacao em Saude: Random Forest vs Rede Neural

**Disciplina:** Inteligencia Computacional - UFPA/PPGEE 2026.1  
**Dataset:** Sleep Health and Daily Performance Dataset  
**Target principal:** `felt_rested`  

Este notebook centraliza a trilha completa do trabalho: verificacao dos dados, EDA, pre-processamento, treinamento dos modelos, avaliacao e comparacao final. Os scripts numerados continuam servindo como pipeline auditavel; o notebook funciona como camada unica de analise e narrativa.

## Objetivos analiticos

1. Verificar qualidade basica do dataset.
2. Explorar distribuicoes, correlacoes e relacoes com `felt_rested`.
3. Treinar Random Forest com ajuste de hiperparametros.
4. Treinar Rede Neural MLP baseline.
5. Treinar Rede Neural MLP otimizada com engenharia de atributos.
6. Comparar os modelos usando as metricas obrigatorias: acuracia, matriz de confusao, precisao, recall, F1-score e ROC AUC.

In [ ]:
from pathlib import Path
import json
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import BatchNormalization, Dense, Dropout, Input
from tensorflow.keras.models import Sequential

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 7)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "sleep_health_dataset.csv"
REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"
MODELS_DIR = PROJECT_ROOT / "models"
NOTEBOOK_MODELS_DIR = MODELS_DIR / "notebook"

for path in [FIGURES_DIR, NOTEBOOK_MODELS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.20
RUN_TRAINING = True
FAST_MODE = False

np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

print(f"Projeto: {PROJECT_ROOT}")
print(f"Dataset: {DATA_PATH}")

## 1. Carregamento e verificacao dos dados

In [ ]:
df = pd.read_csv(DATA_PATH)

display(df.head())
print(f"Linhas: {df.shape[0]:,}")
print(f"Colunas: {df.shape[1]:,}")
print(f"Duplicatas: {df.duplicated().sum():,}")
print(f"Valores nulos: {df.isna().sum().sum():,}")

target_distribution = df["felt_rested"].value_counts(normalize=True).rename("proporcao")
display(target_distribution.to_frame())

In [ ]:
display(df.describe(include="all").T)

## 2. Analise exploratoria de dados

In [ ]:
numeric_df = df.select_dtypes(include=["float64", "int64"])
corr = numeric_df.corr()

plt.figure(figsize=(15, 10))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Mapa de Calor de Correlacao - Variaveis Numericas")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "notebook_correlation_heatmap.png", dpi=150)
plt.show()

target_corr = corr["felt_rested"].sort_values(ascending=False)
display(target_corr.to_frame("correlacao_com_felt_rested").head(10))
display(target_corr.to_frame("correlacao_com_felt_rested").tail(10))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.ravel()

sns.histplot(data=df, x="age", hue="felt_rested", multiple="stack", kde=True, ax=axes[0])
axes[0].set_title("Idade por sensacao de descanso")

sns.boxplot(data=df, x="felt_rested", y="sleep_quality_score", ax=axes[1])
axes[1].set_title("Qualidade do sono vs descanso")

sns.boxplot(data=df, x="felt_rested", y="stress_score", ax=axes[2])
axes[2].set_title("Estresse vs descanso")

sns.boxplot(data=df, x="felt_rested", y="work_hours_that_day", ax=axes[3])
axes[3].set_title("Horas trabalhadas vs descanso")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "notebook_eda_sleep_stress_work.png", dpi=150)
plt.show()

In [ ]:
occupation_rest = pd.crosstab(df["occupation"], df["felt_rested"], normalize="index") * 100
occupation_rest = occupation_rest.sort_values(by=1, ascending=True)

occupation_rest.plot(kind="barh", stacked=True, figsize=(12, 8))
plt.title("Percentual de descanso percebido por ocupacao")
plt.xlabel("Percentual (%)")
plt.ylabel("Ocupacao")
plt.legend(title="Sentiu-se descansado?", labels=["Nao", "Sim"])
plt.tight_layout()
plt.savefig(FIGURES_DIR / "notebook_occupation_vs_target.png", dpi=150)
plt.show()

## 3. Preparacao dos dados

Removemos identificador e targets secundarios para evitar vazamento conceitual. A comparacao usa `felt_rested` como alvo binario.

In [ ]:
DROP_COLUMNS = ["person_id", "felt_rested", "sleep_disorder_risk", "cognitive_performance_score"]

X = df.drop(columns=DROP_COLUMNS)
y = df["felt_rested"]

categorical_features = X.select_dtypes(include=["object", "str"]).columns.tolist()
numeric_features = X.select_dtypes(include=["float64", "int64"]).columns.tolist()

print("Features categoricas:", categorical_features)
print("Features numericas:", numeric_features)

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Treino: {X_train_raw.shape}")
print(f"Teste: {X_test_raw.shape}")

In [ ]:
def make_preprocessor(X_frame):
    categorical = X_frame.select_dtypes(include=["object", "str"]).columns.tolist()
    numeric = X_frame.select_dtypes(include=["float64", "int64"]).columns.tolist()
    return ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numeric),
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical),
        ]
    )


def evaluate_binary_model(name, y_true, y_pred, y_proba):
    metrics = {
        "modelo": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, y_proba),
    }
    print(f"\n### {name} ###")
    print(json.dumps(metrics, indent=2))
    print("\nMatriz de confusao:")
    print(confusion_matrix(y_true, y_pred))
    print("\nRelatorio de classificacao:")
    print(classification_report(y_true, y_pred))
    return metrics


def plot_confusion_matrix(cm, title):
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
    plt.title(title)
    plt.xlabel("Predito")
    plt.ylabel("Real")
    plt.show()

## 4. Modelo 1 - Random Forest com GridSearchCV

In [ ]:
rf_metrics = None

if RUN_TRAINING:
    rf_pipeline = Pipeline(
        steps=[
            ("preprocessor", make_preprocessor(X_train_raw)),
            ("classifier", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)),
        ]
    )

    rf_param_grid = {
        "classifier__n_estimators": [100] if FAST_MODE else [100, 200],
        "classifier__max_depth": [20] if FAST_MODE else [None, 10, 20],
        "classifier__min_samples_split": [2] if FAST_MODE else [2, 5],
    }

    rf_search = GridSearchCV(
        rf_pipeline,
        rf_param_grid,
        cv=3,
        scoring="f1",
        n_jobs=-1,
        verbose=1,
    )
    rf_search.fit(X_train_raw, y_train)
    best_rf = rf_search.best_estimator_

    rf_pred = best_rf.predict(X_test_raw)
    rf_proba = best_rf.predict_proba(X_test_raw)[:, 1]
    rf_metrics = evaluate_binary_model("Random Forest", y_test, rf_pred, rf_proba)

    print("Melhores parametros:", rf_search.best_params_)
    joblib.dump(best_rf, NOTEBOOK_MODELS_DIR / "random_forest_notebook.pkl")
else:
    print("RUN_TRAINING=False. Pule esta celula para apenas revisar narrativa e EDA.")

## 5. Modelo 2 - Rede Neural MLP baseline

In [ ]:
def create_baseline_mlp(input_dim):
    model = Sequential(
        [
            Input(shape=(input_dim,)),
            Dense(64, activation="relu"),
            Dropout(0.2),
            Dense(32, activation="relu"),
            Dropout(0.1),
            Dense(16, activation="relu"),
            Dense(1, activation="sigmoid"),
        ]
    )
    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.AUC(name="auc")],
    )
    return model


nn_metrics = None

if RUN_TRAINING:
    nn_preprocessor = make_preprocessor(X_train_raw)
    X_train_nn = nn_preprocessor.fit_transform(X_train_raw)
    X_test_nn = nn_preprocessor.transform(X_test_raw)

    baseline_mlp = create_baseline_mlp(X_train_nn.shape[1])
    baseline_history = baseline_mlp.fit(
        X_train_nn,
        y_train,
        validation_split=0.2,
        epochs=50 if not FAST_MODE else 5,
        batch_size=128,
        callbacks=[EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)],
        verbose=1,
    )

    nn_proba = baseline_mlp.predict(X_test_nn).ravel()
    nn_pred = (nn_proba > 0.5).astype(int)
    nn_metrics = evaluate_binary_model("Rede Neural MLP", y_test, nn_pred, nn_proba)

    baseline_mlp.save(NOTEBOOK_MODELS_DIR / "neural_network_baseline_notebook.keras")
    joblib.dump(baseline_history.history, NOTEBOOK_MODELS_DIR / "nn_baseline_history_notebook.pkl")

## 6. Modelo 3 - Rede Neural MLP otimizada

A segunda rede adiciona engenharia de atributos e uma arquitetura mais profunda. A hipotese e que interacoes entre estresse, carga de trabalho, idade e eficiencia do sono podem melhorar o sinal para a rede.

In [ ]:
def add_sleep_features(frame):
    enriched = frame.copy()
    enriched["stress_work_interaction"] = enriched["stress_score"] * enriched["work_hours_that_day"]
    enriched["sleep_efficiency"] = (enriched["rem_percentage"] + enriched["deep_sleep_percentage"]) / 100
    enriched["age_stress_ratio"] = enriched["stress_score"] / (enriched["age"] + 1)
    return enriched


def create_optimized_mlp(input_dim):
    model = Sequential(
        [
            Input(shape=(input_dim,)),
            Dense(128, activation="relu"),
            BatchNormalization(),
            Dropout(0.3),
            Dense(64, activation="relu"),
            BatchNormalization(),
            Dropout(0.2),
            Dense(32, activation="relu"),
            BatchNormalization(),
            Dense(16, activation="relu"),
            Dense(1, activation="sigmoid"),
        ]
    )
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.AUC(name="auc")],
    )
    return model


nn_opt_metrics = None

if RUN_TRAINING:
    X_train_opt_raw = add_sleep_features(X_train_raw)
    X_test_opt_raw = add_sleep_features(X_test_raw)

    opt_preprocessor = make_preprocessor(X_train_opt_raw)
    X_train_opt = opt_preprocessor.fit_transform(X_train_opt_raw)
    X_test_opt = opt_preprocessor.transform(X_test_opt_raw)

    optimized_mlp = create_optimized_mlp(X_train_opt.shape[1])
    optimized_history = optimized_mlp.fit(
        X_train_opt,
        y_train,
        validation_split=0.2,
        epochs=100 if not FAST_MODE else 5,
        batch_size=64,
        callbacks=[
            EarlyStopping(monitor="val_loss", patience=12, restore_best_weights=True),
            ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-5),
        ],
        verbose=1,
    )

    nn_opt_proba = optimized_mlp.predict(X_test_opt).ravel()
    nn_opt_pred = (nn_opt_proba > 0.5).astype(int)
    nn_opt_metrics = evaluate_binary_model("Rede Neural MLP Otimizada", y_test, nn_opt_pred, nn_opt_proba)

    optimized_mlp.save(NOTEBOOK_MODELS_DIR / "neural_network_optimized_notebook.keras")
    joblib.dump(optimized_history.history, NOTEBOOK_MODELS_DIR / "nn_optimized_history_notebook.pkl")

## 7. Comparacao final

In [ ]:
historical_metrics = [
    {"modelo": "Random Forest (scripts)", "accuracy": 0.7415, "precision": np.nan, "recall": np.nan, "f1": 0.66, "roc_auc": 0.8218},
    {"modelo": "Rede Neural MLP (scripts)", "accuracy": 0.7323, "precision": np.nan, "recall": np.nan, "f1": 0.64, "roc_auc": 0.8142},
    {"modelo": "Rede Neural MLP Otimizada (scripts)", "accuracy": 0.7336, "precision": np.nan, "recall": np.nan, "f1": np.nan, "roc_auc": 0.8145},
]

current_metrics = [m for m in [rf_metrics, nn_metrics, nn_opt_metrics] if m is not None]
comparison_df = pd.DataFrame(current_metrics if current_metrics else historical_metrics)
display(comparison_df)

plot_df = comparison_df.set_index("modelo")[["accuracy", "f1", "roc_auc"]]
plot_df.plot(kind="bar", figsize=(12, 6))
plt.title("Comparacao dos Modelos")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "notebook_model_comparison.png", dpi=150)
plt.show()

In [ ]:
if RUN_TRAINING:
    plt.figure(figsize=(10, 7))
    for label, proba in [
        ("Random Forest", locals().get("rf_proba")),
        ("Rede Neural MLP", locals().get("nn_proba")),
        ("Rede Neural MLP Otimizada", locals().get("nn_opt_proba")),
    ]:
        if proba is None:
            continue
        fpr, tpr, _ = roc_curve(y_test, proba)
        auc = roc_auc_score(y_test, proba)
        plt.plot(fpr, tpr, label=f"{label} (AUC={auc:.4f})")

    plt.plot([0, 1], [0, 1], "--", color="gray")
    plt.title("Curvas ROC")
    plt.xlabel("Taxa de Falsos Positivos")
    plt.ylabel("Taxa de Verdadeiros Positivos")
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "notebook_roc_curves.png", dpi=150)
    plt.show()

## 8. Sintese para o relatorio

A Random Forest apresentou o melhor desempenho geral no conjunto tabular analisado. Mesmo com engenharia de atributos, BatchNormalization, Dropout e ajuste de learning rate, a rede neural otimizada melhorou apenas marginalmente em relacao ao seu baseline e nao superou a Random Forest.

A interpretacao principal e que, para este dataset estruturado com variaveis numericas e categoricas, modelos baseados em arvores conseguem capturar interacoes nao lineares de forma eficiente, com menor custo de ajuste. A rede neural permanece competitiva, mas exige mais decisoes de arquitetura e regularizacao para ganho pequeno.

Ponto importante para a defesa: o resultado nao indica que redes neurais sao inferiores de forma geral; indica que, neste problema tabular especifico, a Random Forest foi a escolha mais robusta e pragmatica.